# Verdict Oracle — one-time verified Drive import

Run as the project owner; your professor will not run this. The notebook updates your ZIP with the **existing retained-judgment text cache**, not the full raw 1,384-PDF folder. Only verified training cases are included.

In [ ]:
!pip -q install google-api-python-client google-auth-httplib2 pandas

In [ ]:
from google.colab import files
from pathlib import Path
import shutil,zipfile,subprocess,sys
print('Upload Verdict_Oracle_Final_Application.zip')
uploaded=files.upload()
zipname=next((x for x in uploaded if x.endswith('.zip')),None)
if not zipname: raise ValueError('Upload the application ZIP')
base=Path('/content/verdict_oracle_release');shutil.rmtree(base,ignore_errors=True);base.mkdir()
with zipfile.ZipFile(zipname) as z:z.extractall(base)
appfolder=base/'Verdict_Oracle_Final_Application'
if not appfolder.exists():raise FileNotFoundError('Expected Verdict_Oracle_Final_Application inside ZIP')
print('Application extracted:',appfolder)

In [ ]:
# Run the importer in this notebook's Python process, so Drive authentication is shared.
from google.colab import auth
auth.authenticate_user()
import importlib.util, traceback
print('Importing verified judgment texts from Google Drive...')
spec = importlib.util.spec_from_file_location('verdict_drive_import', appfolder / 'import_verified_drive_cache.py')
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
try:
    report = module.import_from_drive()
except Exception:
    print('The full error is below. Please copy it if you need help troubleshooting.')
    traceback.print_exc()
    raise


In [ ]:
import json
report_path=appfolder/'data'/'pdf_coverage_report.json'
if not report_path.exists():
    raise FileNotFoundError('No coverage report was generated. Review the full error in the previous cell.')
report=json.loads(report_path.read_text(encoding='utf-8'))
print('Verified training cases indexed:',report['indexed_training_pdfs'])
print('Reasons for other cases:',report['matching_status_counts'])
if report['indexed_training_pdfs'] == 0:
    raise RuntimeError('No cases were matched; do not deploy. Review the citation matching audit.')


In [ ]:
result=Path('/content/Verdict_Oracle_Deploy_Ready.zip')
with zipfile.ZipFile(result,'w',zipfile.ZIP_DEFLATED) as z:
    for p in appfolder.rglob('*'):
        if p.is_file() and '__pycache__' not in p.parts:
            z.write(p,p.relative_to(appfolder))
print('Deployment ZIP created:',result,'bytes:',result.stat().st_size)
files.download(str(result))
